# Other Log Types

Covers ModuleLog, ModuleCallLog, ParamLog, BufferLog, GradFnLog, and GradFnCallLog records.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, has_trainable_params=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_grads=True,
    save_function_args=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].out.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.ops.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.ops.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer ops, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## Other Log Types: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.BufferLog",
    "tl.BufferLog.DEFAULT_FILL_STATE",
    "tl.BufferLog.FORK_POLICY",
    "tl.BufferLog.PORTABLE_STATE_SPEC",
    "tl.BufferLog.out_transform",
    "tl.BufferLog.buffer_address",
    "tl.BufferLog.buffer_parent",
    "tl.BufferLog.buffer_pass",
    "tl.BufferLog.co_parents",
    "tl.BufferLog.module",
    "tl.BufferLog.modules",
    "tl.BufferLog.copy",
    "tl.BufferLog.get_children",
    "tl.BufferLog.get_parents",
    "tl.BufferLog.grad_dtype",
    "tl.BufferLog.grad_memory",
    "tl.BufferLog.grad_memory_str",
    "tl.BufferLog.grad_shape",
    "tl.BufferLog.has_co_parents",
    "tl.BufferLog.has_grad",
    "tl.BufferLog.has_parents",
    "tl.BufferLog.has_saved_outs",
    "tl.BufferLog.has_siblings",
    "tl.BufferLog.in_submodule",
    "tl.BufferLog.is_submodule_input",
    "tl.BufferLog.layer_label",
    "tl.BufferLog.log_tensor_grad",
    "tl.BufferLog.macs_backward",
    "tl.BufferLog.macs_forward",
    "tl.BufferLog.materialize_out",
    "tl.BufferLog.materialize_grad",
    "tl.BufferLog.address",
    "tl.BufferLog.module_call_depth",
    "tl.BufferLog.name",
    "tl.BufferLog.num_param_tensors",
    "tl.BufferLog.params",
    "tl.BufferLog.param_memory_str",
    "tl.BufferLog.call_index",
    "tl.BufferLog.ops",
    "tl.BufferLog.save_tensor_data",
    "tl.BufferLog.show",
    "tl.BufferLog.siblings",
    "tl.BufferLog.source_trace",
    "tl.BufferLog.tensor",
    "tl.BufferLog.dtype",
    "tl.BufferLog.memory",
    "tl.BufferLog.memory_str",
    "tl.BufferLog.shape",
    "tl.BufferLog.uses_params",
    "tl.GradFnLog",
    "tl.GradFnLog.PORTABLE_STATE_SPEC",
    "tl.GradFnLog.op",
    "tl.GradFnLog.grad_fn_id",
    "tl.GradFnLog.grad_fn_total_num",
    "tl.GradFnLog.grad_fn_type",
    "tl.GradFnLog.grad_fn_type_num",
    "tl.GradFnLog.is_custom",
    "tl.GradFnLog.has_op",
    "tl.GradFnLog.label",
    "tl.GradFnLog.log_pass",
    "tl.GradFnLog.module_path",
    "tl.GradFnLog.name",
    "tl.GradFnLog.next_grad_fn_ids",
    "tl.GradFnLog.num_calls",
    "tl.GradFnLog.call_labels",
    "tl.GradFnLog.ops",
    "tl.GradFnLog.to_pandas",
    "tl.GradFnCallLog",
    "tl.GradFnCallLog.PORTABLE_STATE_SPEC",
    "tl.GradFnCallLog.grad_inputs",
    "tl.GradFnCallLog.grad_outputs",
    "tl.GradFnCallLog.call_index",
    "tl.GradFnCallLog.time_finished",
    "tl.GradFnCallLog.time_started",
    "tl.GradFnCallLog.to_pandas",
    "tl.ModuleLog",
    "tl.ModuleLog.PORTABLE_STATE_SPEC",
    "tl.ModuleLog.address",
    "tl.ModuleLog.address_children",
    "tl.ModuleLog.address_depth",
    "tl.ModuleLog.address_parent",
    "tl.ModuleLog.all_addresses",
    "tl.ModuleLog.layers",
    "tl.ModuleLog.buffer_layers",
    "tl.ModuleLog.buffers",
    "tl.ModuleLog.call_children",
    "tl.ModuleLog.call_parent",
    "tl.ModuleLog.class_docstring",
    "tl.ModuleLog.custom_attributes",
    "tl.ModuleLog.flops",
    "tl.ModuleLog.flops_backward",
    "tl.ModuleLog.flops_forward",
    "tl.ModuleLog.forward_args",
    "tl.ModuleLog.forward_docstring",
    "tl.ModuleLog.forward_kwargs",
    "tl.ModuleLog.forward_signature",
    "tl.ModuleLog.grad",
    "tl.ModuleLog.has_backward_hooks",
    "tl.ModuleLog.has_forward_hooks",
    "tl.ModuleLog.init_docstring",
    "tl.ModuleLog.init_signature",
    "tl.ModuleLog.input_layers",
    "tl.ModuleLog.inputs",
    "tl.ModuleLog.is_shared",
    "tl.ModuleLog.is_train_mode",
    "tl.ModuleLog.layers",
    "tl.ModuleLog.macs",
    "tl.ModuleLog.macs_backward",
    "tl.ModuleLog.macs_forward",
    "tl.ModuleLog.custom_methods",
    "tl.ModuleLog.class_name",
    "tl.ModuleLog.name",
    "tl.ModuleLog.call_depth",
    "tl.ModuleLog.num_layers",
    "tl.ModuleLog.num_params",
    "tl.ModuleLog.num_params_frozen",
    "tl.ModuleLog.num_params_trainable",
    "tl.ModuleLog.num_calls",
    "tl.ModuleLog.output_layers",
    "tl.ModuleLog.outputs",
    "tl.ModuleLog.params",
    "tl.ModuleLog.param_memory",
    "tl.ModuleLog.param_memory_str",
    "tl.ModuleLog.call_labels",
    "tl.ModuleLog.ops",
    "tl.ModuleLog.has_trainable_params",
    "tl.ModuleLog.show_graph",
    "tl.ModuleLog.source_file",
    "tl.ModuleLog.source_line",
    "tl.ModuleLog.to_csv",
    "tl.ModuleLog.to_json",
    "tl.ModuleLog.to_pandas",
    "tl.ModuleLog.to_parquet",
    "tl.ModuleCallLog",
    "tl.ModuleCallLog.PORTABLE_STATE_SPEC",
    "tl.ModuleCallLog.all_addresses",
    "tl.ModuleCallLog.call_children",
    "tl.ModuleCallLog.call_parent",
    "tl.ModuleCallLog.forward_args",
    "tl.ModuleCallLog.forward_args_summary",
    "tl.ModuleCallLog.forward_kwargs",
    "tl.ModuleCallLog.forward_kwargs_summary",
    "tl.ModuleCallLog.input_layers",
    "tl.ModuleCallLog.inputs",
    "tl.ModuleCallLog.is_shared",
    "tl.ModuleCallLog.layers",
    "tl.ModuleCallLog.address",
    "tl.ModuleCallLog.num_layers",
    "tl.ModuleCallLog.output_layers",
    "tl.ModuleCallLog.outputs",
    "tl.ModuleCallLog.call_label",
    "tl.ModuleCallLog.call_index",
    "tl.ModuleCallLog.to_csv",
    "tl.ModuleCallLog.to_json",
    "tl.ModuleCallLog.to_pandas",
    "tl.ModuleCallLog.to_parquet",
    "tl.ParamLog",
    "tl.ParamLog.PORTABLE_STATE_SPEC",
    "tl.ParamLog.address",
    "tl.ParamLog.barcode",
    "tl.ParamLog.dtype",
    "tl.ParamLog.grad_dtype",
    "tl.ParamLog.grad_memory",
    "tl.ParamLog.grad_memory_str",
    "tl.ParamLog.grad_shape",
    "tl.ParamLog.has_grad",
    "tl.ParamLog.has_optimizer",
    "tl.ParamLog.is_quantized",
    "tl.ParamLog.linked_params",
    "tl.ParamLog.memory",
    "tl.ParamLog.memory_str",
    "tl.ParamLog.address",
    "tl.ParamLog.module_type",
    "tl.ParamLog.name",
    "tl.ParamLog.num_params",
    "tl.ParamLog.num_calls",
    "tl.ParamLog.release_param_ref",
    "tl.ParamLog.shape",
    "tl.ParamLog.trainable",
    "tl.ParamLog.used_by_layers",
    "tl.accessors.GradFnAccessor",
    "tl.accessors.LayerAccessor",
    "tl.accessors.ModuleAccessor",
    "tl.data_classes.BufferAccessor",
    "tl.data_classes.BufferLog",
    "tl.data_classes.FuncCallLocation",
    "tl.data_classes.FuncExecutionContext",
    "tl.data_classes.GradFnAccessor",
    "tl.data_classes.GradFnLog",
    "tl.data_classes.GradFnCallLog",
    "tl.data_classes.ModuleAccessor",
    "tl.data_classes.ModuleLog",
    "tl.data_classes.ModuleCallLog",
    "tl.data_classes.ParamAccessor",
    "tl.data_classes.ParamLog",
    "tl.data_classes.VisualizationOverrides",
    "tl.data_classes.buffer_log",
    "tl.data_classes.cleanup",
    "tl.data_classes.func_call_location",
    "tl.data_classes.grad_fn_log",
    "tl.data_classes.grad_fn_call_log",
    "tl.data_classes.interface",
    "tl.data_classes.internal_types",
    "tl.data_classes.layer_log",
    "tl.data_classes.op_log",
    "tl.data_classes.trace",
    "tl.data_classes.module_log",
    "tl.data_classes.param_log",
    "tl.types.ActivationPostfunc",
    "tl.types.BufferLog",
    "tl.types.FuncCallLocation",
    "tl.types.GradFnLog",
    "tl.types.GradFnCallLog",
    "tl.types.GradientPostfunc",
    "tl.types.ModuleLog",
    "tl.types.ModuleCallLog",
    "tl.types.ParamLog",
    "tl.types.SaveLevel",
    "tl.types.SiteTable",
    "tl.types.SpecCompat",
    "tl.types.TargetManifestDiff",
    "tl.types.TensorLog",
    "tl.types.TensorSliceSpec",
]

for name in ["ModuleLog", "ModuleCallLog", "ParamLog", "BufferLog", "GradFnLog", "GradFnCallLog"]:
    obj = AUDIT_CONTEXT[name]
    print(name, summarize_value(obj))

## ModuleLog manifest

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.ModuleLog",
    "tl.ModuleLog.PORTABLE_STATE_SPEC",
    "tl.ModuleLog.address",
    "tl.ModuleLog.address_children",
    "tl.ModuleLog.address_depth",
    "tl.ModuleLog.address_parent",
    "tl.ModuleLog.all_addresses",
    "tl.ModuleLog.layers",
    "tl.ModuleLog.buffer_layers",
    "tl.ModuleLog.buffers",
    "tl.ModuleLog.call_children",
    "tl.ModuleLog.call_parent",
    "tl.ModuleLog.class_docstring",
    "tl.ModuleLog.custom_attributes",
    "tl.ModuleLog.flops",
    "tl.ModuleLog.flops_backward",
    "tl.ModuleLog.flops_forward",
    "tl.ModuleLog.forward_args",
    "tl.ModuleLog.forward_docstring",
    "tl.ModuleLog.forward_kwargs",
    "tl.ModuleLog.forward_signature",
    "tl.ModuleLog.grad",
    "tl.ModuleLog.has_backward_hooks",
    "tl.ModuleLog.has_forward_hooks",
    "tl.ModuleLog.init_docstring",
    "tl.ModuleLog.init_signature",
    "tl.ModuleLog.input_layers",
    "tl.ModuleLog.inputs",
    "tl.ModuleLog.is_shared",
    "tl.ModuleLog.is_train_mode",
    "tl.ModuleLog.layers",
    "tl.ModuleLog.macs",
    "tl.ModuleLog.macs_backward",
    "tl.ModuleLog.macs_forward",
    "tl.ModuleLog.custom_methods",
    "tl.ModuleLog.class_name",
    "tl.ModuleLog.name",
    "tl.ModuleLog.call_depth",
    "tl.ModuleLog.num_layers",
    "tl.ModuleLog.num_params",
    "tl.ModuleLog.num_params_frozen",
    "tl.ModuleLog.num_params_trainable",
    "tl.ModuleLog.num_calls",
    "tl.ModuleLog.output_layers",
    "tl.ModuleLog.outputs",
    "tl.ModuleLog.params",
    "tl.ModuleLog.param_memory",
    "tl.ModuleLog.param_memory_str",
    "tl.ModuleLog.call_labels",
    "tl.ModuleLog.ops",
    "tl.ModuleLog.has_trainable_params",
    "tl.ModuleLog.show_graph",
    "tl.ModuleLog.source_file",
    "tl.ModuleLog.source_line",
    "tl.ModuleLog.to_csv",
    "tl.ModuleLog.to_json",
    "tl.ModuleLog.to_pandas",
    "tl.ModuleLog.to_parquet",
]

audit_touch_items("ModuleLog manifest", ITEMS, AUDIT_CONTEXT)

## ModuleCallLog manifest

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.ModuleCallLog",
    "tl.ModuleCallLog.PORTABLE_STATE_SPEC",
    "tl.ModuleCallLog.all_addresses",
    "tl.ModuleCallLog.call_children",
    "tl.ModuleCallLog.call_parent",
    "tl.ModuleCallLog.forward_args",
    "tl.ModuleCallLog.forward_args_summary",
    "tl.ModuleCallLog.forward_kwargs",
    "tl.ModuleCallLog.forward_kwargs_summary",
    "tl.ModuleCallLog.input_layers",
    "tl.ModuleCallLog.inputs",
    "tl.ModuleCallLog.is_shared",
    "tl.ModuleCallLog.layers",
    "tl.ModuleCallLog.address",
    "tl.ModuleCallLog.num_layers",
    "tl.ModuleCallLog.output_layers",
    "tl.ModuleCallLog.outputs",
    "tl.ModuleCallLog.call_label",
    "tl.ModuleCallLog.call_index",
    "tl.ModuleCallLog.to_csv",
    "tl.ModuleCallLog.to_json",
    "tl.ModuleCallLog.to_pandas",
    "tl.ModuleCallLog.to_parquet",
]

audit_touch_items("ModuleCallLog manifest", ITEMS, AUDIT_CONTEXT)

## ParamLog manifest

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.ParamLog",
    "tl.ParamLog.PORTABLE_STATE_SPEC",
    "tl.ParamLog.address",
    "tl.ParamLog.barcode",
    "tl.ParamLog.dtype",
    "tl.ParamLog.grad_dtype",
    "tl.ParamLog.grad_memory",
    "tl.ParamLog.grad_memory_str",
    "tl.ParamLog.grad_shape",
    "tl.ParamLog.has_grad",
    "tl.ParamLog.has_optimizer",
    "tl.ParamLog.is_quantized",
    "tl.ParamLog.linked_params",
    "tl.ParamLog.memory",
    "tl.ParamLog.memory_str",
    "tl.ParamLog.address",
    "tl.ParamLog.module_type",
    "tl.ParamLog.name",
    "tl.ParamLog.num_params",
    "tl.ParamLog.num_calls",
    "tl.ParamLog.release_param_ref",
    "tl.ParamLog.shape",
    "tl.ParamLog.trainable",
    "tl.ParamLog.used_by_layers",
]

audit_touch_items("ParamLog manifest", ITEMS, AUDIT_CONTEXT)

## BufferLog manifest

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.BufferLog",
    "tl.BufferLog.DEFAULT_FILL_STATE",
    "tl.BufferLog.FORK_POLICY",
    "tl.BufferLog.PORTABLE_STATE_SPEC",
    "tl.BufferLog.out_transform",
    "tl.BufferLog.buffer_address",
    "tl.BufferLog.buffer_parent",
    "tl.BufferLog.buffer_pass",
    "tl.BufferLog.co_parents",
    "tl.BufferLog.module",
    "tl.BufferLog.modules",
    "tl.BufferLog.copy",
    "tl.BufferLog.get_children",
    "tl.BufferLog.get_parents",
    "tl.BufferLog.grad_dtype",
    "tl.BufferLog.grad_memory",
    "tl.BufferLog.grad_memory_str",
    "tl.BufferLog.grad_shape",
    "tl.BufferLog.has_co_parents",
    "tl.BufferLog.has_grad",
    "tl.BufferLog.has_parents",
    "tl.BufferLog.has_saved_outs",
    "tl.BufferLog.has_siblings",
    "tl.BufferLog.in_submodule",
    "tl.BufferLog.is_submodule_input",
    "tl.BufferLog.layer_label",
    "tl.BufferLog.log_tensor_grad",
    "tl.BufferLog.macs_backward",
    "tl.BufferLog.macs_forward",
    "tl.BufferLog.materialize_out",
    "tl.BufferLog.materialize_grad",
    "tl.BufferLog.address",
    "tl.BufferLog.module_call_depth",
    "tl.BufferLog.name",
    "tl.BufferLog.num_param_tensors",
    "tl.BufferLog.params",
    "tl.BufferLog.param_memory_str",
    "tl.BufferLog.call_index",
    "tl.BufferLog.ops",
    "tl.BufferLog.save_tensor_data",
    "tl.BufferLog.show",
    "tl.BufferLog.siblings",
    "tl.BufferLog.source_trace",
    "tl.BufferLog.tensor",
    "tl.BufferLog.dtype",
    "tl.BufferLog.memory",
    "tl.BufferLog.memory_str",
    "tl.BufferLog.shape",
    "tl.BufferLog.uses_params",
]

audit_touch_items("BufferLog manifest", ITEMS, AUDIT_CONTEXT)

## GradFnLog manifest

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.GradFnLog",
    "tl.GradFnLog.PORTABLE_STATE_SPEC",
    "tl.GradFnLog.op",
    "tl.GradFnLog.grad_fn_id",
    "tl.GradFnLog.grad_fn_total_num",
    "tl.GradFnLog.grad_fn_type",
    "tl.GradFnLog.grad_fn_type_num",
    "tl.GradFnLog.is_custom",
    "tl.GradFnLog.has_op",
    "tl.GradFnLog.label",
    "tl.GradFnLog.log_pass",
    "tl.GradFnLog.module_path",
    "tl.GradFnLog.name",
    "tl.GradFnLog.next_grad_fn_ids",
    "tl.GradFnLog.num_calls",
    "tl.GradFnLog.call_labels",
    "tl.GradFnLog.ops",
    "tl.GradFnLog.to_pandas",
]

audit_touch_items("GradFnLog manifest", ITEMS, AUDIT_CONTEXT)

## GradFnCallLog manifest

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.GradFnCallLog",
    "tl.GradFnCallLog.PORTABLE_STATE_SPEC",
    "tl.GradFnCallLog.grad_inputs",
    "tl.GradFnCallLog.grad_outputs",
    "tl.GradFnCallLog.call_index",
    "tl.GradFnCallLog.time_finished",
    "tl.GradFnCallLog.time_started",
    "tl.GradFnCallLog.to_pandas",
]

audit_touch_items("GradFnCallLog manifest", ITEMS, AUDIT_CONTEXT)

## Other log namespaces

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.accessors.GradFnAccessor",
    "tl.accessors.LayerAccessor",
    "tl.accessors.ModuleAccessor",
    "tl.data_classes.BufferAccessor",
    "tl.data_classes.BufferLog",
    "tl.data_classes.FuncCallLocation",
    "tl.data_classes.FuncExecutionContext",
    "tl.data_classes.GradFnAccessor",
    "tl.data_classes.GradFnLog",
    "tl.data_classes.GradFnCallLog",
    "tl.data_classes.ModuleAccessor",
    "tl.data_classes.ModuleLog",
    "tl.data_classes.ModuleCallLog",
    "tl.data_classes.ParamAccessor",
    "tl.data_classes.ParamLog",
    "tl.data_classes.VisualizationOverrides",
    "tl.data_classes.buffer_log",
    "tl.data_classes.cleanup",
    "tl.data_classes.func_call_location",
    "tl.data_classes.grad_fn_log",
    "tl.data_classes.grad_fn_call_log",
    "tl.data_classes.interface",
    "tl.data_classes.internal_types",
    "tl.data_classes.layer_log",
    "tl.data_classes.op_log",
    "tl.data_classes.trace",
    "tl.data_classes.module_log",
    "tl.data_classes.param_log",
    "tl.types.ActivationPostfunc",
    "tl.types.BufferLog",
    "tl.types.FuncCallLocation",
    "tl.types.GradFnLog",
    "tl.types.GradFnCallLog",
    "tl.types.GradientPostfunc",
    "tl.types.ModuleLog",
    "tl.types.ModuleCallLog",
    "tl.types.ParamLog",
    "tl.types.SaveLevel",
    "tl.types.SiteTable",
    "tl.types.SpecCompat",
    "tl.types.TargetManifestDiff",
    "tl.types.TensorLog",
    "tl.types.TensorSliceSpec",
]

audit_touch_items("Other log namespaces", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.BufferLog",
    "tl.BufferLog.DEFAULT_FILL_STATE",
    "tl.BufferLog.FORK_POLICY",
    "tl.BufferLog.PORTABLE_STATE_SPEC",
    "tl.BufferLog.out_transform",
    "tl.BufferLog.buffer_address",
    "tl.BufferLog.buffer_parent",
    "tl.BufferLog.buffer_pass",
    "tl.BufferLog.co_parents",
    "tl.BufferLog.module",
    "tl.BufferLog.modules",
    "tl.BufferLog.copy",
    "tl.BufferLog.get_children",
    "tl.BufferLog.get_parents",
    "tl.BufferLog.grad_dtype",
    "tl.BufferLog.grad_memory",
    "tl.BufferLog.grad_memory_str",
    "tl.BufferLog.grad_shape",
    "tl.BufferLog.has_co_parents",
    "tl.BufferLog.has_grad",
    "tl.BufferLog.has_parents",
    "tl.BufferLog.has_saved_outs",
    "tl.BufferLog.has_siblings",
    "tl.BufferLog.in_submodule",
    "tl.BufferLog.is_submodule_input",
    "tl.BufferLog.layer_label",
    "tl.BufferLog.log_tensor_grad",
    "tl.BufferLog.macs_backward",
    "tl.BufferLog.macs_forward",
    "tl.BufferLog.materialize_out",
    "tl.BufferLog.materialize_grad",
    "tl.BufferLog.address",
    "tl.BufferLog.module_call_depth",
    "tl.BufferLog.name",
    "tl.BufferLog.num_param_tensors",
    "tl.BufferLog.params",
    "tl.BufferLog.param_memory_str",
    "tl.BufferLog.call_index",
    "tl.BufferLog.ops",
    "tl.BufferLog.save_tensor_data",
    "tl.BufferLog.show",
    "tl.BufferLog.siblings",
    "tl.BufferLog.source_trace",
    "tl.BufferLog.tensor",
    "tl.BufferLog.dtype",
    "tl.BufferLog.memory",
    "tl.BufferLog.memory_str",
    "tl.BufferLog.shape",
    "tl.BufferLog.uses_params",
    "tl.GradFnLog",
    "tl.GradFnLog.PORTABLE_STATE_SPEC",
    "tl.GradFnLog.op",
    "tl.GradFnLog.grad_fn_id",
    "tl.GradFnLog.grad_fn_total_num",
    "tl.GradFnLog.grad_fn_type",
    "tl.GradFnLog.grad_fn_type_num",
    "tl.GradFnLog.is_custom",
    "tl.GradFnLog.has_op",
    "tl.GradFnLog.label",
    "tl.GradFnLog.log_pass",
    "tl.GradFnLog.module_path",
    "tl.GradFnLog.name",
    "tl.GradFnLog.next_grad_fn_ids",
    "tl.GradFnLog.num_calls",
    "tl.GradFnLog.call_labels",
    "tl.GradFnLog.ops",
    "tl.GradFnLog.to_pandas",
    "tl.GradFnCallLog",
    "tl.GradFnCallLog.PORTABLE_STATE_SPEC",
    "tl.GradFnCallLog.grad_inputs",
    "tl.GradFnCallLog.grad_outputs",
    "tl.GradFnCallLog.call_index",
    "tl.GradFnCallLog.time_finished",
    "tl.GradFnCallLog.time_started",
    "tl.GradFnCallLog.to_pandas",
    "tl.ModuleLog",
    "tl.ModuleLog.PORTABLE_STATE_SPEC",
    "tl.ModuleLog.address",
    "tl.ModuleLog.address_children",
    "tl.ModuleLog.address_depth",
    "tl.ModuleLog.address_parent",
    "tl.ModuleLog.all_addresses",
    "tl.ModuleLog.layers",
    "tl.ModuleLog.buffer_layers",
    "tl.ModuleLog.buffers",
    "tl.ModuleLog.call_children",
    "tl.ModuleLog.call_parent",
    "tl.ModuleLog.class_docstring",
    "tl.ModuleLog.custom_attributes",
    "tl.ModuleLog.flops",
    "tl.ModuleLog.flops_backward",
    "tl.ModuleLog.flops_forward",
    "tl.ModuleLog.forward_args",
    "tl.ModuleLog.forward_docstring",
    "tl.ModuleLog.forward_kwargs",
    "tl.ModuleLog.forward_signature",
    "tl.ModuleLog.grad",
    "tl.ModuleLog.has_backward_hooks",
    "tl.ModuleLog.has_forward_hooks",
    "tl.ModuleLog.init_docstring",
    "tl.ModuleLog.init_signature",
    "tl.ModuleLog.input_layers",
    "tl.ModuleLog.inputs",
    "tl.ModuleLog.is_shared",
    "tl.ModuleLog.is_train_mode",
    "tl.ModuleLog.layers",
    "tl.ModuleLog.macs",
    "tl.ModuleLog.macs_backward",
    "tl.ModuleLog.macs_forward",
    "tl.ModuleLog.custom_methods",
    "tl.ModuleLog.class_name",
    "tl.ModuleLog.name",
    "tl.ModuleLog.call_depth",
    "tl.ModuleLog.num_layers",
    "tl.ModuleLog.num_params",
    "tl.ModuleLog.num_params_frozen",
    "tl.ModuleLog.num_params_trainable",
    "tl.ModuleLog.num_calls",
    "tl.ModuleLog.output_layers",
    "tl.ModuleLog.outputs",
    "tl.ModuleLog.params",
    "tl.ModuleLog.param_memory",
    "tl.ModuleLog.param_memory_str",
    "tl.ModuleLog.call_labels",
    "tl.ModuleLog.ops",
    "tl.ModuleLog.has_trainable_params",
    "tl.ModuleLog.show_graph",
    "tl.ModuleLog.source_file",
    "tl.ModuleLog.source_line",
    "tl.ModuleLog.to_csv",
    "tl.ModuleLog.to_json",
    "tl.ModuleLog.to_pandas",
    "tl.ModuleLog.to_parquet",
    "tl.ModuleCallLog",
    "tl.ModuleCallLog.PORTABLE_STATE_SPEC",
    "tl.ModuleCallLog.all_addresses",
    "tl.ModuleCallLog.call_children",
    "tl.ModuleCallLog.call_parent",
    "tl.ModuleCallLog.forward_args",
    "tl.ModuleCallLog.forward_args_summary",
    "tl.ModuleCallLog.forward_kwargs",
    "tl.ModuleCallLog.forward_kwargs_summary",
    "tl.ModuleCallLog.input_layers",
    "tl.ModuleCallLog.inputs",
    "tl.ModuleCallLog.is_shared",
    "tl.ModuleCallLog.layers",
    "tl.ModuleCallLog.address",
    "tl.ModuleCallLog.num_layers",
    "tl.ModuleCallLog.output_layers",
    "tl.ModuleCallLog.outputs",
    "tl.ModuleCallLog.call_label",
    "tl.ModuleCallLog.call_index",
    "tl.ModuleCallLog.to_csv",
    "tl.ModuleCallLog.to_json",
    "tl.ModuleCallLog.to_pandas",
    "tl.ModuleCallLog.to_parquet",
    "tl.ParamLog",
    "tl.ParamLog.PORTABLE_STATE_SPEC",
    "tl.ParamLog.address",
    "tl.ParamLog.barcode",
    "tl.ParamLog.dtype",
    "tl.ParamLog.grad_dtype",
    "tl.ParamLog.grad_memory",
    "tl.ParamLog.grad_memory_str",
    "tl.ParamLog.grad_shape",
    "tl.ParamLog.has_grad",
    "tl.ParamLog.has_optimizer",
    "tl.ParamLog.is_quantized",
    "tl.ParamLog.linked_params",
    "tl.ParamLog.memory",
    "tl.ParamLog.memory_str",
    "tl.ParamLog.address",
    "tl.ParamLog.module_type",
    "tl.ParamLog.name",
    "tl.ParamLog.num_params",
    "tl.ParamLog.num_calls",
    "tl.ParamLog.release_param_ref",
    "tl.ParamLog.shape",
    "tl.ParamLog.trainable",
    "tl.ParamLog.used_by_layers",
    "tl.accessors.GradFnAccessor",
    "tl.accessors.LayerAccessor",
    "tl.accessors.ModuleAccessor",
    "tl.data_classes.BufferAccessor",
    "tl.data_classes.BufferLog",
    "tl.data_classes.FuncCallLocation",
    "tl.data_classes.FuncExecutionContext",
    "tl.data_classes.GradFnAccessor",
    "tl.data_classes.GradFnLog",
    "tl.data_classes.GradFnCallLog",
    "tl.data_classes.ModuleAccessor",
    "tl.data_classes.ModuleLog",
    "tl.data_classes.ModuleCallLog",
    "tl.data_classes.ParamAccessor",
    "tl.data_classes.ParamLog",
    "tl.data_classes.VisualizationOverrides",
    "tl.data_classes.buffer_log",
    "tl.data_classes.cleanup",
    "tl.data_classes.func_call_location",
    "tl.data_classes.grad_fn_log",
    "tl.data_classes.grad_fn_call_log",
    "tl.data_classes.interface",
    "tl.data_classes.internal_types",
    "tl.data_classes.layer_log",
    "tl.data_classes.op_log",
    "tl.data_classes.trace",
    "tl.data_classes.module_log",
    "tl.data_classes.param_log",
    "tl.types.ActivationPostfunc",
    "tl.types.BufferLog",
    "tl.types.FuncCallLocation",
    "tl.types.GradFnLog",
    "tl.types.GradFnCallLog",
    "tl.types.GradientPostfunc",
    "tl.types.ModuleLog",
    "tl.types.ModuleCallLog",
    "tl.types.ParamLog",
    "tl.types.SaveLevel",
    "tl.types.SiteTable",
    "tl.types.SpecCompat",
    "tl.types.TargetManifestDiff",
    "tl.types.TensorLog",
    "tl.types.TensorSliceSpec",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")